In [ ]:
from pathlib import Path
from fastkaggle import iskaggle, setup_comp

# Set these only when you want to send your results to your Kaggle account.
submit_to_kaggle = False
publish_to_kaggle = False

## Memory and gradient accumulation

In [ ]:
comp = 'paddy-disease-classification'
path = Path(comp)
if iskaggle or not (path/'train_images').is_dir():
    path = setup_comp(comp)
from fastai.vision.all import *
set_seed(42)

tst_files = get_image_files(path/'test_images').sorted()

In [ ]:
df = pd.read_csv(path/'train.csv')
df.label.value_counts()

In [ ]:
trn_path = path/'train_images'/'bacterial_panicle_blight'

In [ ]:
def train(arch, size, item=Resize(480, method='squish'), accum=1, finetune=True, epochs=12):
    dls = ImageDataLoaders.from_folder(trn_path, valid_pct=0.2, item_tfms=item,
        batch_tfms=aug_transforms(size=size, min_scale=0.75), bs=64//accum)
    cbs = GradientAccumulation(64) if accum else []
    learn = vision_learner(dls, arch, metrics=error_rate, cbs=cbs)
    # CUDA uses the lecture precision. The CPU kernel uses float32.
    if torch.cuda.is_available(): learn = learn.to_fp16()
    if finetune:
        learn.fine_tune(epochs, 0.01)
        return learn.tta(dl=dls.test_dl(tst_files))
    else:
        learn.unfreeze()
        learn.fit_one_cycle(epochs, 0.01)

In [ ]:
train('convnext_small.fb_in22k', 128, epochs=1, accum=1, finetune=False)

In [ ]:
import gc
def report_gpu():
    if torch.cuda.is_available(): print(torch.cuda.list_gpu_processes())
    else: print("GPU memory measurements require a CUDA kernel.")
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
report_gpu()

In [ ]:
train('convnext_small.fb_in22k', 128, epochs=1, accum=2, finetune=False)
report_gpu()

In [ ]:
train('convnext_small.fb_in22k', 128, epochs=1, accum=4, finetune=False)
report_gpu()

## Checking memory use

In [ ]:
train('convnext_large.fb_in22k', 224, epochs=1, accum=2, finetune=False)
report_gpu()

In [ ]:
train('convnext_large.fb_in22k', (320,240), epochs=1, accum=2, finetune=False)
report_gpu()

In [ ]:
train('vit_large_patch16_224.augreg_in21k_ft_in1k', 224, epochs=1, accum=2, finetune=False)
report_gpu()

In [ ]:
train('swinv2_large_window12_192.ms_in22k', 192, epochs=1, accum=2, finetune=False)
report_gpu()

In [ ]:
train('swin_large_patch4_window7_224.ms_in22k_ft_in1k', 224, epochs=1, accum=2, finetune=False)
report_gpu()

## Running the models

In [ ]:
res = 640,480

In [ ]:
models = {
    'convnext_large.fb_in22k': {
        (Resize(res), 224),
        (Resize(res), (320,224)),
    }, 'vit_large_patch16_224.augreg_in21k_ft_in1k': {
        (Resize(480, method='squish'), 224),
        (Resize(res), 224),
    }, 'swinv2_large_window12_192.ms_in22k': {
        (Resize(480, method='squish'), 192),
        (Resize(res), 192),
    }, 'swin_large_patch4_window7_224.ms_in22k_ft_in1k': {
        (Resize(480, method='squish'), 224),
        (Resize(res), 224),
    }
}

In [ ]:
trn_path = path/'train_images'

In [ ]:
tta_res = []

for arch,details in models.items():
    for item,size in details:
        print('---',arch)
        print(size)
        print(item.name)
        tta_res.append(train(arch, size, item=item, accum=2)) #, epochs=1))
        gc.collect()
        torch.cuda.empty_cache()

## Ensembling

In [ ]:
save_pickle('tta_res.pkl', tta_res)

In [ ]:
tta_prs = first(zip(*tta_res))

In [ ]:
tta_prs += tta_prs[2:4]

In [ ]:
avg_pr = torch.stack(tta_prs).mean(0)
avg_pr.shape

In [ ]:
dls = ImageDataLoaders.from_folder(trn_path, valid_pct=0.2, item_tfms=Resize(480, method='squish'),
    batch_tfms=aug_transforms(size=224, min_scale=0.75))

In [ ]:
idxs = avg_pr.argmax(dim=1)
vocab = np.array(dls.vocab)
ss = pd.read_csv(path/'sample_submission.csv')
ss['label'] = vocab[idxs]
ss.to_csv('subm.csv', index=False)

In [ ]:
if submit_to_kaggle and not iskaggle:
    from kaggle import api
    api.competition_submit('subm.csv', 'part 3 v2', comp)

## Conclusion

In [ ]:
# Optional: run from the directory containing this notebook.
if publish_to_kaggle and not iskaggle:
    import json, shutil
    from tempfile import TemporaryDirectory
    from fastkaggle import nb_meta
    from kaggle import api

    filename = '10-scaling-up-road-to-the-top-part-3.ipynb'
    meta = nb_meta(api.config_values['username'], 'scaling-up-road-to-the-top-part-3', 'Scaling Up: Road to the Top, Part 3',
                   filename, competition=comp, private=False, gpu=True)
    with TemporaryDirectory() as folder:
        shutil.copyfile(filename, Path(folder)/filename)
        (Path(folder)/'kernel-metadata.json').write_text(json.dumps(meta), encoding='utf-8')
        api.kernels_push(folder)